# Advanced tutorial: Compare urban and forested watersheds

## Research question

**Are summer water temperatures different between predominantly developed and predominantly forested STREAM watersheds?**

## `streamwq` functions used

- `stream.get_sites()`
- `stream.get_data("land_cover", ...)`
- `stream.get_data("water_quality", ...)`
- `filter_season()`
- `resample_timeseries()`

In [ ]:
import streamwq as stream
from streamwq.helpers.pandas import filter_season, resample_timeseries

## 1. Read land cover for temperature-candidate stations

In [ ]:
sites = stream.get_sites(variables="WTemp_C")
developed_columns = [
    "Developed Open Space", "Developed Low Intensity",
    "Developed Medium Intensity", "Developed High Intensity",
]
forest_columns = ["Deciduous Forest", "Evergreen Forest", "Mixed Forest"]
land_cover = stream.get_data(
    "land_cover",
    site_ids=sites,
    columns=["STREAM_ID", "DateTime", *(developed_columns + forest_columns)],
).to_pandas()
if land_cover.empty:
    raise ValueError("No land-cover observations were returned for these candidate sites")

## 2. Define comparison groups

These 50% thresholds are transparent tutorial choices, not universal scientific definitions. The most recent land-cover record is used for each station.

In [ ]:
latest = (
    land_cover.sort_values("DateTime")
    .groupby("STREAM_ID", as_index=False)
    .tail(1)
    .copy()
)
latest["developed_fraction"] = latest[developed_columns].sum(axis=1)
latest["forest_fraction"] = latest[forest_columns].sum(axis=1)
land_cover_scale = 1.0 if latest[developed_columns + forest_columns].max().max() <= 1 else 100.0
majority_threshold = 0.50 * land_cover_scale
latest["watershed_group"] = "other"
latest.loc[latest["developed_fraction"] > majority_threshold, "watershed_group"] = "developed"
latest.loc[latest["forest_fraction"] > majority_threshold, "watershed_group"] = "forested"
latest["watershed_group"].value_counts()

## 3. Calculate station-level summer temperature

In [ ]:
if not comparison_ids:
    raise ValueError("No stations had enough land-cover data to classify as developed or forested")
temperature = stream.get_data(
    "water_quality",
    site_ids=comparison_ids,
    columns=["STREAM_ID", "DateTime", "WTemp_C"],
    quality="approved",
).to_pandas()
if temperature.empty:
    raise ValueError("No approved water-temperature observations matched the comparison sites")
summer = filter_season(temperature, "summer")
daily_summer = resample_timeseries(summer, "WTemp_C", "daily", "mean")
station_means = daily_summer.groupby("STREAM_ID", as_index=False)["WTemp_C"].mean()
analysis = station_means.merge(
    latest[["STREAM_ID", "watershed_group"]], on="STREAM_ID"
)
analysis.groupby("watershed_group")["WTemp_C"].describe()